# Silver → Gold

Gold **không làm sạch gì thêm**: chỉ JOIN fact với dimension rồi GROUP BY bằng Spark SQL. Mỗi bảng
Silver được đăng ký thành một temp view cùng tên.

In [1]:
import csv, json, logging, shutil, sys
from pathlib import Path

import pandas as pd

# Thư mục gốc dự án = nơi có src/config.py (kernel có thể mở với cwd bất kỳ)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
for p in (ROOT, ROOT / "tests"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import bronze, config, gold, silver

logging.basicConfig(level=logging.INFO, format="%(levelname)-5s %(name)-7s %(message)s",
                    stream=sys.stdout, force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)
pd.set_option("display.max_columns", 40)

SANDBOX = ROOT / "notebook" / "_sandbox"             # mọi thứ notebook ghi ra nằm ở đây
SB_BRONZE = SANDBOX / "bronze" / "business_valuation"
SB_CONTROL = SANDBOX / "control" / "loaded_batches.csv"
SB_SILVER = SANDBOX / "silver"
SB_GOLD = SANDBOX / "gold"

# Số đo trên 18 lô thật trong tài liệu — để SO, không assert (dữ liệu đổi thì số đổi)
EXPECTED = {"marts": {"mart_vehicle_price": 951, "mart_monthly_trend": 876,
                     "mart_data_quality": 18, "mart_ml_training": 44_748}}

Hàm tiện ích dùng trong notebook này:

In [2]:
def fresh(*paths: Path) -> None:
    """Xoá các thư mục/file con của sandbox — notebook chỉ dọn phần của mình."""
    for p in paths:
        if SANDBOX not in p.parents:
            raise ValueError(f"{p} nằm ngoài sandbox — không xoá")
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists():
            p.unlink()
    SANDBOX.mkdir(parents=True, exist_ok=True)


def show(df, n: int = 10) -> pd.DataFrame:
    """DataFrame Spark -> pandas để notebook hiển thị thành bảng."""
    return df.limit(n).toPandas()


def compare(actual: dict, expected: dict) -> pd.DataFrame:
    """Bảng so số thực tế với số đo ghi trong tài liệu."""
    keys = list(expected) + [k for k in actual if k not in expected]
    return pd.DataFrame([{"mục": k, "thực tế": actual.get(k, 0), "tài liệu": expected.get(k, "—"),
                          "khớp": actual.get(k, 0) == expected.get(k)} for k in keys])


def ensure_bronze(spark) -> None:
    """Sandbox chưa có Bronze (chưa chạy notebook 02) -> nạp 18 lô."""
    if not SB_BRONZE.exists():
        print("Sandbox chưa có Bronze → nạp 18 lô (như notebook 02)")
        bronze.run(spark, config.LANDING_DIR, SB_BRONZE, SB_CONTROL)


def ensure_silver(spark) -> None:
    """Sandbox chưa có Silver (chưa chạy notebook 03) -> dựng Bronze nếu cần, rồi Silver."""
    ensure_bronze(spark)
    if not (SB_SILVER / "fact_valuation").exists():
        print("Sandbox chưa có Silver → chạy Silver (như notebook 03)")
        silver.run(spark, SB_BRONZE, SB_SILVER)


fresh(SB_GOLD)

In [3]:
# khởi động Spark
from pyspark.sql import functions as F
from src.spark import get_spark

spark = get_spark("04_gold")
print("Spark", spark.version)

Spark 3.5.9


In [4]:
# chưa có Silver trong sandbox thì dựng Bronze → Silver
ensure_silver(spark)

# Bronze active và fact: dùng để đối soát số dòng ở các bước sau
active = silver.active_bronze(spark.read.parquet(SB_BRONZE.as_posix()))
fact = spark.read.parquet((SB_SILVER / "fact_valuation").as_posix())
print(f"Bronze active: {active.count()} dòng · fact: {fact.count()} dòng")

Bronze active: 52791 dòng · fact: 44748 dòng


In [5]:
# mỗi bảng Silver thành một temp view cùng tên -> viết SQL: FROM fact_valuation JOIN dim_model ...
gold.register_silver(spark, SB_SILVER)
print("Temp view:", [t.name for t in spark.catalog.listTables()])

Temp view: ['dim_date', 'dim_model', 'dim_vehicle_type', 'fact_valuation', 'quarantine']


In [6]:
# kiểm tra: đọc qua view và đọc thẳng parquet phải ra cùng số dòng
n_view = spark.sql("SELECT COUNT(*) AS n FROM fact_valuation").first()["n"]
assert n_view == fact.count()
n_view

44748

## `mart_vehicle_price`: giá trung vị mới/cũ và tỷ lệ còn lại ρ

- Grain: 1 dòng = 1 nhóm (hãng, dòng xe, năm sản xuất, số chỗ).
- Giá xe mới và xe cũ tính riêng bằng `MEDIAN(CASE WHEN ...)`. Trung vị không bị vài giá cực đoan kéo lệch.
- ρ = giá cũ / giá mới. Nhóm chỉ có một phía (chỉ có xe mới hoặc chỉ có xe cũ) thì ρ là null.

In [7]:
# câu SQL: fact JOIN dim_model, dim_date rồi GROUP BY thuộc tính dimension
print(gold.MART_VEHICLE_PRICE)


WITH g AS (
    SELECT m.hang_xe, m.hieu_xe, f.nam_san_xuat, f.so_cho_ngoi,
           COUNT(*)                                                             AS so_hop_dong,
           COUNT_IF(f.loai_dinh_gia = 'new')                                    AS so_hd_xe_moi,
           COUNT_IF(f.loai_dinh_gia = 'used')                                   AS so_hd_xe_cu,
           MEDIAN(CASE WHEN f.loai_dinh_gia = 'new'  THEN f.gia_tri_trieu_vnd END) AS gia_xe_moi_trung_vi,
           MEDIAN(CASE WHEN f.loai_dinh_gia = 'used' THEN f.gia_tri_trieu_vnd END) AS gia_xe_cu_trung_vi,
           MIN(d.nam_thang)                                                     AS thang_dau_tien,
           MAX(d.nam_thang)                                                     AS thang_gan_nhat
    FROM fact_valuation f
    JOIN dim_model m ON f.model_key      = m.model_key
    JOIN dim_date  d ON f.ngay_duyet_key = d.date_key
    GROUP BY m.hang_xe, m.hieu_xe, f.nam_san_xuat, f.so_cho_ngoi
)
SELECT hang_xe, hieu_x

In [8]:
mvp = spark.sql(gold.MART_VEHICLE_PRICE).cache()
print(mvp.count(), "nhóm")
show(mvp, 5)

951 nhóm


,hang_xe,hieu_xe,nam_san_xuat,so_cho_ngoi,so_hop_dong,so_hd_xe_moi,so_hd_xe_cu,gia_xe_moi_trung_vi,gia_xe_cu_trung_vi,ty_le_con_lai,thang_dau_tien,thang_gan_nhat
0,BYD,Seal 5,2024,7,5,1,4,726.0,756.0,1.041,2025-01,2025-05
1,BYD,Seal 5,2025,1,1,1,0,756.0,NaN,NaN,2025-02,2025-02
2,BYD,Seal 5,2025,5,4,3,1,756.0,700.0,0.926,2025-02,2026-06
3,BYD,Seal 5,2025,7,128,111,17,756.0,700.0,0.926,2025-01,2026-06
4,BYD,Seal 5,2026,6,2,2,0,746.0,NaN,NaN,2026-06,2026-06


In [9]:
# kiểm tra grain: mỗi (hãng, dòng, năm sx, số chỗ) chỉ có 1 dòng
grain = ["hang_xe", "hieu_xe", "nam_san_xuat", "so_cho_ngoi"]
assert mvp.groupBy(grain).count().filter("count > 1").count() == 0, "Trùng grain"

# kiểm tra JOIN + GROUP BY không mất hợp đồng: tổng so_hop_dong = số dòng fact
tong = mvp.agg(F.sum("so_hop_dong"), F.sum("so_hd_xe_moi"), F.sum("so_hd_xe_cu")).first()
assert tong[0] == fact.count(), "JOIN làm mất hoặc nhân dòng"
assert tong[1] + tong[2] == tong[0], "Có hợp đồng không phải new/used"
print(f"tổng hợp đồng {tong[0]} = fact {fact.count()} · mới {tong[1]} + cũ {tong[2]}")

tổng hợp đồng 44748 = fact 44748 · mới 15665 + cũ 29083


In [10]:
# chỉ giữ nhóm có đủ giá mới & cũ -> tính được ρ
rho = mvp.filter(F.col("ty_le_con_lai").isNotNull())
print(f"đủ cả giá mới & cũ: {rho.count()} nhóm · "
      f"trung vị ρ: {rho.select(F.median('ty_le_con_lai')).first()[0]} · "
      f"ρ > 1: {rho.filter('ty_le_con_lai > 1').count()}")
print("Tài liệu: 115 nhóm đủ hai phía · trung vị ρ 0,938 · 13 nhóm ρ > 1")

đủ cả giá mới & cũ: 115 nhóm · trung vị ρ: 0.938 · ρ > 1: 13
Tài liệu: 115 nhóm đủ hai phía · trung vị ρ 0,938 · 13 nhóm ρ > 1


In [11]:
# 10 nhóm nhiều hợp đồng nhất
show(rho.orderBy(F.desc("so_hop_dong")), 10)

,hang_xe,hieu_xe,nam_san_xuat,so_cho_ngoi,so_hop_dong,so_hd_xe_moi,so_hd_xe_cu,gia_xe_moi_trung_vi,gia_xe_cu_trung_vi,ty_le_con_lai,thang_dau_tien,thang_gan_nhat
0,VinFast,VF6,2025,5,1999,1456,543,507.8,450.0,0.886,2025-01,2026-06
1,VinFast,VF3,2025,4,1710,1387,323,290.0,270.0,0.931,2025-01,2026-06
2,VinFast,Limo Green,2026,7,989,988,1,689.1,692.1,1.004,2026-01,2026-06
3,VinFast,VF7,2025,5,833,698,135,715.2,661.4,0.925,2025-01,2026-06
4,Mitsubishi,Xpander,2025,7,745,610,135,582.0,560.0,0.962,2025-01,2026-06
5,MG,ZS,2025,5,679,607,72,431.0,410.0,0.951,2025-01,2026-06
6,VinFast,VF6,2024,5,625,31,594,542.7,460.0,0.848,2025-01,2026-06
7,Mitsubishi,Xforce,2025,5,548,461,87,598.0,600.0,1.003,2025-01,2026-06
8,Toyota,Vios,2025,5,493,436,57,487.8,460.0,0.943,2025-01,2026-06
9,VinFast,Limo Green,2025,7,451,375,76,707.0,698.5,0.988,2025-09,2026-06


In [12]:
# ρ > 1: giá xe cũ cao hơn xe mới -> xem so_hd_xe_moi / so_hd_xe_cu trước khi kết luận
show(rho.filter("ty_le_con_lai > 1").orderBy(F.desc("so_hop_dong")), 15)

,hang_xe,hieu_xe,nam_san_xuat,so_cho_ngoi,so_hop_dong,so_hd_xe_moi,so_hd_xe_cu,gia_xe_moi_trung_vi,gia_xe_cu_trung_vi,ty_le_con_lai,thang_dau_tien,thang_gan_nhat
0,VinFast,Limo Green,2026,7,989,988,1,689.1,692.1,1.004,2026-01,2026-06
1,Mitsubishi,Xforce,2025,5,548,461,87,598.0,600.0,1.003,2025-01,2026-06
2,Hyundai,Accent,2025,5,428,379,49,471.0,480.0,1.019,2025-01,2026-06
3,Mazda,3,2025,5,299,242,57,485.0,490.0,1.010,2025-01,2026-06
4,Toyota,Vios,2024,5,213,3,210,438.0,450.0,1.027,2025-01,2026-06
5,Ford,Everest,2025,7,178,130,48,1262.0,1269.0,1.006,2025-01,2026-06
6,Hyundai,Creta,2026,5,140,126,14,624.5,675.8,1.082,2026-01,2026-06
7,Hyundai,Tucson,2025,5,122,99,23,843.0,850.0,1.008,2025-01,2026-06
8,Honda,HRV,2025,5,55,37,18,664.1,666.5,1.004,2025-01,2026-06
9,Kia,Carnival,2026,7,26,25,1,1524.0,1686.0,1.106,2026-01,2026-06


## `mart_monthly_trend`

- Grain: (tháng duyệt, hãng, mục đích sử dụng, loại định giá).
- JOIN đủ 3 dimension: tháng từ `dim_date`, hãng từ `dim_model`, mục đích sử dụng từ `dim_vehicle_type`.
- Đo: số hợp đồng, giá trung vị, tuổi xe trung bình.

In [13]:
print(gold.MART_MONTHLY_TREND)


SELECT d.nam_thang, m.hang_xe, v.muc_dich_su_dung, f.loai_dinh_gia,
       COUNT(*)                              AS so_hop_dong,
       ROUND(MEDIAN(f.gia_tri_trieu_vnd), 1) AS gia_trung_vi,
       ROUND(AVG(f.tuoi_xe), 1)              AS tuoi_xe_trung_binh
FROM fact_valuation f
JOIN dim_date         d ON f.ngay_duyet_key   = d.date_key
JOIN dim_model        m ON f.model_key        = m.model_key
JOIN dim_vehicle_type v ON f.vehicle_type_key = v.vehicle_type_key
GROUP BY d.nam_thang, m.hang_xe, v.muc_dich_su_dung, f.loai_dinh_gia
ORDER BY d.nam_thang, m.hang_xe, v.muc_dich_su_dung, f.loai_dinh_gia



In [14]:
mmt = spark.sql(gold.MART_MONTHLY_TREND).cache()
print(mmt.count(), "dòng")
show(mmt, 10)

876 dòng


,nam_thang,hang_xe,muc_dich_su_dung,loai_dinh_gia,so_hop_dong,gia_trung_vi,tuoi_xe_trung_binh
0,2025-01,BYD,Không kinh doanh,new,1,756.0,0.0
1,2025-01,BYD,Không kinh doanh,used,1,756.0,1.0
2,2025-01,Ford,Không kinh doanh,new,11,855.0,0.0
3,2025-01,Ford,Không kinh doanh,used,80,800.0,4.2
4,2025-01,Ford,Kinh doanh vận tải,used,9,700.0,4.2
5,2025-01,Honda,Không kinh doanh,new,14,650.0,0.1
6,2025-01,Honda,Không kinh doanh,used,148,515.0,5.0
7,2025-01,Honda,Kinh doanh vận tải,used,2,750.0,2.5
8,2025-01,Hyundai,Không kinh doanh,new,64,524.5,0.1
9,2025-01,Hyundai,Không kinh doanh,used,367,450.0,4.2


In [15]:
# kiểm tra JOIN 3 dimension không mất dòng: tổng so_hop_dong = số dòng fact
assert mmt.agg(F.sum("so_hop_dong")).first()[0] == fact.count()

In [16]:
# gộp theo tháng để xem xu hướng tổng số hợp đồng
show(mmt.groupBy("nam_thang").agg(F.sum("so_hop_dong").alias("so_hop_dong")).orderBy("nam_thang"), 30)

,nam_thang,so_hop_dong
0,2025-01,2681
1,2025-02,1634
2,2025-03,2208
3,2025-04,2018
4,2025-05,2029
5,2025-06,2079
6,2025-07,2303
7,2025-08,1967
8,2025-09,1930
9,2025-10,2991


## `mart_data_quality`: mỗi tháng bao nhiêu dòng vào fact / quarantine

- `UNION ALL` fact (gán `rule_id` = null) với quarantine, rồi GROUP BY theo tháng.
- Mỗi luật một cột đếm. Danh sách cột sinh từ `silver.RULES`, không viết tay ở hai nơi.

In [17]:
# SQL sinh từ silver.RULES -> thêm luật ở Silver là mart tự có thêm cột
print(gold.mart_data_quality_sql())


WITH all_rows AS (
    SELECT as_of_month, CAST(NULL AS STRING) AS rule_id FROM fact_valuation
    UNION ALL
    SELECT as_of_month, rule_id FROM quarantine
)
SELECT as_of_month,
       COUNT(*)                                                     AS so_dong_bronze,
       COUNT_IF(rule_id IS NULL)                                    AS so_dong_fact,
       COUNT_IF(rule_id IS NOT NULL)                                AS so_dong_quarantine,
       ROUND(100 * COUNT_IF(rule_id IS NOT NULL) / COUNT(*), 2)     AS ty_le_quarantine_pct,
       COUNT_IF(rule_id = 'CORRUPT_RECORD') AS corrupt_record,
       COUNT_IF(rule_id = 'DUP_KEY') AS dup_key,
       COUNT_IF(rule_id = 'MISSING_FIELD') AS missing_field,
       COUNT_IF(rule_id = 'DATE_INVALID') AS date_invalid,
       COUNT_IF(rule_id = 'OUT_OF_SCOPE') AS out_of_scope,
       COUNT_IF(rule_id = 'MODEL_CODE_UNKNOWN') AS model_code_unknown,
       COUNT_IF(rule_id = 'VALUE_INVALID') AS value_invalid,
       COUNT_IF(rule_id = 'PRICE_INVALID'

In [18]:
dq = spark.sql(gold.mart_data_quality_sql()).toPandas()
rule_cols = [r.lower() for r in silver.RULES]
dq

,as_of_month,so_dong_bronze,so_dong_fact,so_dong_quarantine,ty_le_quarantine_pct,corrupt_record,dup_key,missing_field,date_invalid,out_of_scope,model_code_unknown,value_invalid,price_invalid
0,2025-01,3119,2681,438,14.04,0,0,2,0,0,319,0,117
1,2025-02,1982,1634,348,17.56,0,0,0,6,0,199,0,143
2,2025-03,2650,2208,442,16.68,0,0,3,13,0,260,0,166
3,2025-04,2417,2018,399,16.51,0,0,1,5,0,246,1,146
4,2025-05,2422,2029,393,16.23,0,0,1,6,0,214,0,172
5,2025-06,2507,2079,428,17.07,0,0,2,10,0,274,1,141
6,2025-07,2775,2303,472,17.01,0,0,0,12,0,314,12,134
7,2025-08,2333,1967,366,15.69,0,0,1,11,0,216,11,127
8,2025-09,2279,1930,349,15.31,0,0,0,16,0,235,5,93
9,2025-10,3418,2991,427,12.49,0,0,0,19,0,271,9,128


In [19]:
# kiểm tra "một dòng, một rule_id": tổng các cột luật = số dòng quarantine
assert (dq[rule_cols].sum(axis=1) == dq["so_dong_quarantine"]).all(), "Tổng theo luật ≠ quarantine"

In [20]:
# đối soát với Bronze active theo TỪNG tháng (không chỉ tổng)
per_month = active.groupBy("as_of_month").count().toPandas().rename(columns={"count": "active"})
m = dq[["as_of_month", "so_dong_bronze"]].merge(per_month, on="as_of_month", how="outer")
assert (m["so_dong_bronze"] == m["active"]).all(), "Lệch số dòng ở ít nhất 1 tháng"
assert dq["so_dong_bronze"].sum() == active.count()
m

,as_of_month,so_dong_bronze,active
0,2025-01,3119,3119
1,2025-02,1982,1982
2,2025-03,2650,2650
3,2025-04,2417,2417
4,2025-05,2422,2422
5,2025-06,2507,2507
6,2025-07,2775,2775
7,2025-08,2333,2333
8,2025-09,2279,2279
9,2025-10,3418,3418


## `mart_ml_training`: contract đầu ra

- Grain = 1 hợp đồng, giống fact. **Không** GROUP BY, vì model học từ từng quan sát.
- Chia train/test theo **tháng duyệt** với mốc `ML_TRAIN_END_MONTH`, không chia ngẫu nhiên.
- Contract quy định tên cột, thứ tự cột, kiểu dữ liệu và không có null. Sai contract thì dừng, không ghi.

In [21]:
# contract: (cột, kiểu, vai trò). Thứ tự cột cũng là một phần của contract
pd.DataFrame(gold.ML_TRAINING_CONTRACT, columns=["cột", "kiểu", "vai trò"])

,cột,kiểu,vai trò
0,ma_dinh_gia,string,id
1,hang_xe,string,feature
2,hieu_xe,string,feature
3,nam_san_xuat,int,feature
4,tuoi_xe,int,feature
5,so_cho_ngoi,int,feature
6,muc_dich_su_dung,string,feature
7,loai_dinh_gia,string,feature
8,thang_duyet,string,feature
9,gia_tri_trieu_vnd,double,target


In [22]:
print(gold.mart_ml_training_sql())


SELECT f.ma_dinh_gia,
       m.hang_xe,
       m.hieu_xe,
       f.nam_san_xuat,
       f.tuoi_xe,
       f.so_cho_ngoi,
       v.muc_dich_su_dung,
       f.loai_dinh_gia,
       d.nam_thang                                                     AS thang_duyet,
       f.gia_tri_trieu_vnd,
       CASE WHEN d.nam_thang <= '2026-03' THEN 'train' ELSE 'test' END AS split
FROM fact_valuation f
JOIN dim_model        m ON f.model_key        = m.model_key
JOIN dim_vehicle_type v ON f.vehicle_type_key = v.vehicle_type_key
JOIN dim_date         d ON f.ngay_duyet_key   = d.date_key
ORDER BY f.ma_dinh_gia



In [23]:
ml = spark.sql(gold.mart_ml_training_sql()).cache()
ml.printSchema()
show(ml, 5)

root
 |-- ma_dinh_gia: string (nullable = true)
 |-- hang_xe: string (nullable = true)
 |-- hieu_xe: string (nullable = true)
 |-- nam_san_xuat: integer (nullable = true)
 |-- tuoi_xe: integer (nullable = true)
 |-- so_cho_ngoi: integer (nullable = true)
 |-- muc_dich_su_dung: string (nullable = true)
 |-- loai_dinh_gia: string (nullable = true)
 |-- thang_duyet: string (nullable = true)
 |-- gia_tri_trieu_vnd: double (nullable = true)
 |-- split: string (nullable = false)



,ma_dinh_gia,hang_xe,hieu_xe,nam_san_xuat,tuoi_xe,so_cho_ngoi,muc_dich_su_dung,loai_dinh_gia,thang_duyet,gia_tri_trieu_vnd,split
0,BV-0001d0c75b9b3a9e,Mazda,3,2025,0,5,Không kinh doanh,new,2025-03,639.0,train
1,BV-000374784d1532f5,Toyota,Corolla Cross,2023,2,5,Không kinh doanh,used,2025-05,660.0,train
2,BV-000498d5a8061113,Mitsubishi,Xpander,2024,1,7,Không kinh doanh,used,2025-10,580.0,train
3,BV-0006fb11ec7e5450,Mazda,3,2016,9,5,Không kinh doanh,used,2025-08,300.0,train
4,BV-0008df6924679ae4,Mazda,3,2021,4,5,Không kinh doanh,used,2025-12,520.0,train


In [24]:
# kiểm contract: không ném lỗi = đạt
gold.check_ml_contract(ml)
print("Đạt contract")

Đạt contract


In [25]:
# kiểm tra grain: 1 dòng / hợp đồng, không mất, không nhân dòng
assert ml.count() == fact.count(), "Bảng ML phải đúng 1 dòng / hợp đồng"
assert ml.select("ma_dinh_gia").distinct().count() == ml.count(), "ma_dinh_gia bị trùng"
print(ml.count(), "dòng =", fact.count(), "hợp đồng")

44748 dòng = 44748 hợp đồng


In [26]:
# tóm tắt train/test: số dòng, khoảng tháng, số dòng xe
split = (ml.groupBy("split").agg(F.count("*").alias("so_dong"), F.min("thang_duyet").alias("tu"),
                                 F.max("thang_duyet").alias("den"),
                                 F.countDistinct("hieu_xe").alias("so_dong_xe"))
           .orderBy(F.desc("split")).toPandas())
print("Mốc:", config.ML_TRAIN_END_MONTH, "· tài liệu: train 36.525 · test 8.223")
split

Mốc: 2026-03 · tài liệu: train 36.525 · test 8.223


,split,so_dong,tu,den,so_dong_xe
0,train,36525,2025-01,2026-03,61
1,test,8223,2026-04,2026-06,61


In [27]:
# kiểm tra không tháng nào ở cả train lẫn test, và train <= mốc < test
overlap = ml.groupBy("thang_duyet").agg(F.countDistinct("split").alias("n")).filter("n > 1").count()
assert overlap == 0, "Có tháng nằm ở cả train lẫn test"
s = split.set_index("split")
assert s.loc["train", "den"] <= config.ML_TRAIN_END_MONTH < s.loc["test", "tu"]

In [28]:
# 4 kiểu làm lệch bảng: thiếu cột, sai thứ tự, sai kiểu, target có null
drifts = {
    "thiếu cột split": ml.drop("split"),
    "đổi thứ tự cột": ml.select(*reversed(ml.columns)),
    "tuoi_xe thành bigint": ml.withColumn("tuoi_xe", F.col("tuoi_xe").cast("bigint")),
    "target có null": ml.withColumn("gia_tri_trieu_vnd", F.when(F.col("split") == "train",
                                                                F.col("gia_tri_trieu_vnd"))),
}

In [29]:
# đưa từng bảng lệch qua contract: phải ném ValueError
outcome = []
for name, bad in drifts.items():
    try:
        gold.check_ml_contract(bad)
        outcome.append({"lệch": name, "kết quả": "LỌT — contract không bắt được!"})
    except ValueError as e:
        outcome.append({"lệch": name, "kết quả": "chặn: " + str(e).splitlines()[0]})
outcome = pd.DataFrame(outcome)
outcome

,lệch,kết quả
0,thiếu cột split,chặn: mart_ml_training sai schema:
1,đổi thứ tự cột,chặn: mart_ml_training sai schema:
2,tuoi_xe thành bigint,chặn: mart_ml_training sai schema:
3,target có null,chặn: mart_ml_training có null: {'gia_tri_trie...


In [30]:
assert outcome["kết quả"].str.startswith("chặn").all(), "Có kiểu lệch lọt qua contract"

## Ghi Gold và chạy lại cho ra đúng kết quả cũ

`csv_dir=None`: không xuất CSV, để `output/` thật không bị ghi đè.

In [31]:
# ghi 4 mart; mart_ml_training được kiểm contract TRƯỚC khi ghi
res_gold = gold.run(spark, SB_SILVER, SB_GOLD, csv_dir=None)
compare(res_gold, EXPECTED["marts"])

INFO  gold    Gold: mart_vehicle_price      951 dòng
INFO  gold    Gold: mart_monthly_trend      876 dòng
INFO  gold    Gold: mart_data_quality        18 dòng
INFO  gold    Gold: mart_ml_training      44748 dòng
INFO  gold                               train 36525 · test 8223 (mốc 2026-03)


,mục,thực tế,tài liệu,khớp
0,mart_vehicle_price,951,951,True
1,mart_monthly_trend,876,876,True
2,mart_data_quality,18,18,True
3,mart_ml_training,44748,44748,True


In [32]:
# mỗi mart 1 file (coalesce(1))
for p in sorted(SB_GOLD.rglob("*.parquet")):
    print(p.relative_to(SB_GOLD), f"{p.stat().st_size/1024:.0f} KB")

mart_data_quality\part-00000-1cd34601-73a8-46d7-a92d-b81cd2975ddd-c000.snappy.parquet 5 KB
mart_ml_training\part-00000-15aae559-771b-4816-b3f2-029c599fed64-c000.snappy.parquet 943 KB
mart_monthly_trend\part-00000-8d354b3b-9a74-40e5-8c5b-69c610cc5c6e-c000.snappy.parquet 8 KB
mart_vehicle_price\part-00000-e7450ffc-a3c8-4777-a521-11aef998dd3a-c000.snappy.parquet 15 KB


In [33]:
# kiểm tra số dòng đã ghi
assert res_gold == {"mart_vehicle_price": mvp.count(), "mart_monthly_trend": mmt.count(),
                    "mart_data_quality": len(dq), "mart_ml_training": ml.count()}

# file đã ghi, đọc lại vẫn đạt contract
gold.check_ml_contract(spark.read.parquet((SB_GOLD / "mart_ml_training").as_posix()))

In [34]:
# chạy lại lần 2: full refresh (overwrite) -> ra đúng kết quả cũ, không nhân đôi
res_gold_again = gold.run(spark, SB_SILVER, SB_GOLD, csv_dir=None)
assert res_gold == res_gold_again

INFO  gold    Gold: mart_vehicle_price      951 dòng
INFO  gold    Gold: mart_monthly_trend      876 dòng
INFO  gold    Gold: mart_data_quality        18 dòng
INFO  gold    Gold: mart_ml_training      44748 dòng
INFO  gold                               train 36525 · test 8223 (mốc 2026-03)


In [36]:
spark.stop()
# fresh(SANDBOX / ...)